# പാഠം 13 - കോഗ്നി നോളജ് ഗ്രാഫുകളോടുള്ള ഏജന്റ് മെമ്മറി


## സെറ്റപ്പ്

ഈ നോട്ട്‌ബുക്ക് [**Cognee**](https://www.cognee.ai/) നോളജ് ഗ്രാഫുകളും **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് സ്ഥിരമായ മെമ്മറിയുള്ള ബുദ്ധिमുട്ടായ **കോഡിംഗ് അസിസ്റ്റന്റ്** എങ്ങനെ ഉണ്ടാക്കാമെന്ന് കാണിക്കുന്നു.

Cognee ഘടനരഹിത ടെക്സ്റ്റ് ഘടനാപരവും ക്വെറിയാകാവുന്ന നോളജ് ഗ്രാഫായി പരിവർത്തനം ചെയ്യുന്നു, വെക്ടർ എംബെഡിങ്ങുകൾ പിന്തുണയ്ക്കുന്നു — നിങ്ങളുടെ ഏജന്റിന് സമ്പന്നവും ബന്ധAware ആയ ദീർഘകാല മെമ്മറി നൽകുന്നു.

### നിങ്ങൾ പഠിക്കുന്നതു
1. **നോളജ് ഗ്രാഫുകൾ നിർമ്മിക്കുക**: ഡെവലപ്പർ പ്രൊഫൈലുകളും മികച്ച അനുഭവവുമ ഘടനാപരവും ക്വെറിയാകാവുന്ന വിജ്ഞാനമായി മാറ്റുക.
2. **Cognee-നെ MAF-യുമായി സംയോജിപ്പിക്കുക**: `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് MAF ഏജന്റ് Cognee നോളജ് ഗ്രാഫിൽ ക്വറി ചെയ്യാൻ അനുവദിക്കുക.
3. **സെഷൻ അവബോധമുള്ള സംഭാഷണങ്ങൾ**: ഒരേ സെഷനിൽ പല ചോദ്യങ്ങളിലും സാദൃശ്യ സൂക്ഷിക്കുക.
4. **ദീർഘകാല മെമ്മറി**: സെഷനുകൾക്കിടയിൽ പ്രധാന വിജ്ഞാനവും നിലനിർത്തും, പുതിയ സംഭാഷണങ്ങളിൽ അത് വീണ്ടും തിരിച്ചു വരയുക.

### ആവശ്യങ്ങൾ
- പൈത്തൺ 3.9+
- സെഷൻ മാനേജ്മെന്റിനായി ലോക്കലിൽ റെഡിസ് റണ്ണിംഗ് (`docker run -d -p 6379:6379 redis`)
- LLM API കീ (ഉദാ: OpenAI) — `.env` ലെ `LLM_API_KEY` സെറ്റ് ചെയ്യുക
- `.env` ലെ `CACHING=true` (Cognee സെഷനുകൾക്കു വേണ്ടിയത്)
- Microsoft Foundry പ്രോജക്റ്റ് ഒരു ഡിപ്ലോയ് ചെയ്ത ചാറ്റ് മോഡലിനൊപ്പം
- `.env` ലെ `AZURE_AI_PROJECT_ENDPOINT` & `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ഓർത്തentic ചെയ്തത് (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ഏജന്റ് മെമ്മറി തരങ്ങൾ

ഈ നോട്ട്‌ബുക്ക് പ്രധാന പാഠം 13 നോട്ട്‌ബുക്കിലെ മൂന്ന് മെമ്മറി തരം പരിശോധിക്കുന്നു, എന്നാൽ ദീർഘകാല മെമ്മറി ബാക്ക്എൻഡായി Cognee ഉപയോഗിക്കുന്നു:

| മെമ്മറി തരം | ക്രമീകരണം | ആയുസ്സ് |
|---|---|---|
| **വർക്കിങ്** | `agent.create_session()` (MAF) | ഒരെണ്ണം സംഭാഷണം |
| **ഷോർട്ട്-ടേം** | Cognee സെഷൻ ക്യാഷെ (Redis) | ഒരെണ്ണം സെഷൻ |
| **ലോംഗ്-ടേം** | Cognee വിജ്ഞാന ഗ്രാഫ് + വെക്റ്ററുകൾ | സ്ഥിരം |

### Cognee യുടെ മെമ്മറി ഘടന
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## കോഗ്‌നീ സ്റ്റോറേജ് തയ്യാറാക്കുക


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ഭാഗം 1 — നോളേജ്ബേസ് നിർമ്മാണം

ഞങ്ങളുടെ കോഡിംഗ് അസിസ്റ്റന്റിന് സമഗ്രമായ നോളേജ്ബേസ് സൃഷ്ടിക്കാൻ നാം മൂന്നു തരത്തിലുള്ള ഡാറ്റ സ്വീകരിക്കുന്നു:

1. **ഡെവലപ്പർ പ്രൊഫൈൽ** — വ്യക്തിഗത ക്ഷമതയും സാങ്കേതിക പശ്ചാത്തലവും
2. **പൈത്തൺ ബസ്റ്റ് പ്രാക്ടീസുകൾ** — പ്രായോഗിക നയങ്ങളോടെയുള്ള പൈത്തണിന്റെ സുന്ദരമാധ്യമം (Zen of Python)
3. **ചരിത്ര സംഭാഷണങ്ങൾ** — ഡെവലപ്പർമാരും എഐ അസിസ്റ്റന്റുമിടയിലെ മുൻകാല ചോദ്യോത്തര സെഷനുകൾ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## അറിവ് ഗ്രാഫ് ദൃശ്യവത്കരിക്കുക

Cognee അത് പറ്റിച്ച ഘടകങ്ങളും ബന്ധങ്ങളും ഒരു ഇന്ററാക്ടീവ് HTML ദൃശ്യവത്കരണമായി പ്രദർശിപ്പിക്കാം.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## മെമ്മറി മെമ്പിഫൈ ഉപയോഗിച്ച് സമൃദ്ധമാക്കുക

`memify()` നോളേജ് ഗ്രാഫ് വിശകലനം ചെയ്ത് വിവേകമുള്ള നിയമങ്ങൾ സൃഷ്ടിക്കുന്നു — മാതൃകകൾ, മികച്ച രീതികൾ, ആശയങ്ങൾ തമ്മിലുള്ള ബന്ധങ്ങൾ തിരിച്ചറിയുന്നു.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ഭാഗം 2 — Cognee ടൂളുകളുമായി MAF ഏജന്റ്

ഇനി നമ്മള്‍ ഒരു MAF ഏജന്റ് സൃഷ്ടിക്കുന്നു, അതില്‍ `@tool` ഫങ്ഷനുകള്‍ ഉപയോഗിച്ച് Cognee യുടെ അറിവ് ഗ്രാഫ് ക്വറി ചെയ്യാന്‍ കഴിയും. ഇത് ഏജന്റിന് ഗ്രാഫ് അറിവുള്ള സെമാന്റിക് സെര്‍ച്ച് സവിശേഷതകള്‍ പൂര്ണ്ണമായും ഉപയോഗിക്കുവാന്‍ സഹായിക്കുന്നു, കൂടാതെ സെഷനുകള്‍ വഴി സംവാദ സംബന്ധമായ പ്രസ്ഥാനം നിലനിര്‍ത്തുന്നു.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## സെഷനുകളോടൊപ്പം പ്രവർത്തിക്കുന്ന മെമ്മറി

 ഒരു സെഷനിനുള്ളിൽ പ്രവർത്തിക്കുന്ന മെമ്മറി നൽകുന്നത് `AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിക്കപ്പെടുന്നു) ആണ്. ഏജന്റ് മുമ്പത്തെ സന്ദേശങ്ങളെ തിരിച്ച് നോക്കാനും Cognee-യുടെ ദീർഘകാല അറിവുകളുടെ ഗ്രാഫ് അന്വേഷിക്കാനും കഴിയും.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## പുതിയ സെഷൻ — ദീർഘകാല ഓർമ്മ നിലനിൽക്കുന്നു

പുതിയ സെഷൻ ആരംഭിക്കുമ്പോൾ വർക്കിംഗ് മെമ്മറി ക്ലിയർ ചെയ്യപ്പെടും, എന്നാൽ Cognee നോളജ് ഗ്രാഫ് ഇപ്പോഴും ലഭ്യമാണ്. ഏജന്റ് മുഴുവനായും പുതിയ സംഭാഷണത്തിൽ ഒരേ ദീർഘകാല ജ്ഞാനം പുനഃപ്രാപ്തമാക്കാൻ കഴിയും.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംക്ഷേപം

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ നിർമ്മിച്ചത് **MAF ന്റെ പ്രവർത്തന സ്മൃതി** (`agent.create_session()`) **Cognee ന്റെ ദീർഘകാല അറിവ് ഗ്രാഫ്** ഒന്നിപ്പിക്കുന്ന കോഡിംഗ് അസിസ്റ്റന്റാണ്.

### നിങ്ങൾ പഠിച്ചത്
1. **അറിവ് ഗ്രാഫ് നിർമ്മാണം**: Cognee ഘടകരഹിതമായ എഴുത്ത് സ്വീകരിച്ച് ഒരു ഗ്രാഫും വെക്റ്റർ സ്മൃതിയും നിർമിക്കുന്നു.
2. **memify ഉപയോഗിച്ച് ഗ്രാഫ് സമൃദ്ധീകരണം**: നിലവിലുള്ള ഗ്രാഫിൻ മുകളിൽ നിന്നുള്ള കണ്ടെത്തലുകൾക്കും സമ്പന്നമായ ബന്ധങ്ങൾക്കും.
3. **MAF + Cognee സംയോജനം**: `@tool` ഫംഗ്ഷനുകൾ MAF ഏജന്റുകൾക്ക് Cognee ഗ്രാഫിനെ സ്വാഭാവികമായി ചോദിക്കാനായി അനുവദിക്കുന്നു.
4. **പ്രവർത്തന സ്മൃതി + ദീർഘകാല സ്മൃതി**: `AgentSession` (`agent.create_session()` മുഖേന) സെഷൻ പശ്ചാത്തലം നൽകുമ്പോൾ Cognee സ്ഥിരതയുള്ള അറിവ് നൽകുന്നു.
5. **NodeSets ഉപയോഗിച്ച് ഫിൽട്ടർ ചെയ്ത സർച്**: അറിവ് ഗ്രാഫിന്റെ പ്രത്യേക ഉപസമൂഹങ്ങളെ ലക്ഷ്യംവെക്കുക (ഉദാ: فقط സിദ്ധാന്തങ്ങൾ).

### പ്രധാന നേട്ടങ്ങൾ
- **Cognee** അസംര‍ച്ചിത എഴുത്ത് ഘടകരീതിയുള്ള, ബന്ധമനസ്സിലാക്കുന്ന സ്മാർട്ടായി മാറ്റുന്നു — ഒരു തരം വെക്റ്റർ സ്റ്റോറിനെക്കാൾ ശക്തമാണ്.
- **`@tool` ഫംഗ്ഷനുകൾ** MAF ഏജന്റുകളും പുറം അറിവ് സംവിധാനങ്ങളും പാടില്ലാതെ ബന്ധിപ്പിക്കുന്നു.
- **`AgentSession`** (`agent.create_session()` വഴി) ഒരു സംഭാഷണത്തിനുള്ള പശ്ചാത്തലം ദീർഘകാല അറിവിൽനിന്ന് വേർതിരിക്കുന്നു.
- ആറിവ് ഗ്രാഫ് പല സെഷനുകൾക്കും ഏജന്റുകൾക്കും സേവനം നൽകുന്നു.

### യാഥാർത്ഥ്യ ലോക പ്രയോഗങ്ങൾ
- **ഡെവലപ്പർ കോപിലറ്റുകൾ**: കോഡ് റിവ്യൂ, ഇൻസിഡന്റ് വിശകലനം, ആർക്കിടെക്‌ചർ അസിസ്റ്റന്റുകൾ
- **കസ്റ്റമർ ഫേസിംഗ് കോപിലറ്റുകൾ**: ഉൽപ്പന്ന രേഖകൾ, FAQകൾ, CRM നോട്ടുകൾ എന്നിവയിൽ സഹായിക്കുന്ന ഏജന്റുകൾ
- **ആന്തരിക വിദഗ്ധ കോപിലറ്റുകൾ**: നയം, നിയമം, സുരക്ഷ അസിസ്റ്റന്റുകൾ മാർഗനിർദ്ദേശങ്ങളെക്കുറിച്ച് ആലോചിക്കുന്നു
- **ഏകീകൃത ഡാറ്റ ലെയർകൾ**: ഘടകരഹിതമായും ഘടകമായും ഡാറ്റ ഒരു ചോദ്യപതിക്കാൻ കഴിയുന്ന ഗ്രാഫ് ആയി സംയോജിപ്പിക്കുന്നു

### അടുത്ത് ചെയ്യേണ്ടത്
- Cognee യിൽ കാലക്രമ ബോധം പരീക്ഷിക്കുക
- ഡൊമെയ്ൻ പ്രത്യേക ഗ്രാഫ് ഗുണനിലവാരം നിർവചിക്കാൻ OWL ഓന്റോളജി നിർമിക്കുക
- സമയാനുസരിച്ച് റിട്രീവലിൽ മെച്ചപ്പെടുത്താൻ ഉപയോക്തൃ പ്രതികരണ ലൂപ്പുകൾ ചേർക്കുക
- Cognee സ്മൃതി ലെയർ പങ്കുവെക്കുന്ന ബഹു-ഏജന്റ് സിസ്റ്റത്തിലേക്ക് വ്യാപിപ്പിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
